# Logo Recognition Model Training

**Model:** `google/vit-base-patch16-224` fine-tuned on OpenLogos-32  
**Target:** 32-class logo classification → exported to ONNX + INT8 quantization  
**Runtime:** Google Colab T4 GPU (~2 hours)

## Pipeline Overview
1. Install dependencies
2. Mount Google Drive for model persistence
3. Download OpenLogos-32 from Kaggle
4. Data augmentation & DataLoader setup
5. Fine-tune ViT (frozen backbone → full fine-tune)
6. Evaluate: accuracy + confusion matrix
7. Export to ONNX with dynamic batch
8. INT8 quantize with `onnxruntime.quantization`
9. Benchmark CPU inference speed
10. Save final model to Google Drive

> **Output:** `/content/drive/MyDrive/detectra_models/logo_vit_quantized.onnx`  
> Copy this file to `backend/models/logo_vit_quantized.onnx` in your Detectra project.

## Step 1 — Install Dependencies

Install the required packages not already present on Colab. This includes:
- `transformers` for the ViT model
- `datasets` for data utilities
- `onnx` + `onnxruntime` for export and quantization
- `kaggle` for downloading the dataset

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(
    "transformers==4.40.1",
    "datasets==2.19.1",
    "onnx==1.16.0",
    "onnxruntime==1.18.0",
    "kaggle==1.6.14",
    "timm==0.9.16",        # used by some ViT variants
    "accelerate==0.29.3",  # Trainer acceleration
)

print("All dependencies installed.")

## Step 2 — Mount Google Drive

Models and checkpoints are saved to Google Drive so they persist after the Colab session ends.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/detectra_models"
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

CHECKPOINT_DIR = "/content/logo_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Drive mounted. Output dir: {GDRIVE_OUTPUT_DIR}")

## Step 3 — Download OpenLogos-32 from Kaggle

**Setup:** Upload your `kaggle.json` API token to Colab (from https://www.kaggle.com/settings → API → Create New Token).  
The dataset `victorfrei/openlogos` contains 32 brand logo classes.

> If you've already downloaded the dataset to Drive, set `USE_CACHED = True` to skip the download.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

# ── Kaggle credentials ─────────────────────────────────────────────────────────
# Option A: upload kaggle.json via Colab file picker
# from google.colab import files
# uploaded = files.upload()   # upload kaggle.json

# Option B: copy from Google Drive (recommended for repeat runs)
KAGGLE_JSON_PATH_ON_DRIVE = "/content/drive/MyDrive/kaggle.json"
KAGGLE_DEST = os.path.expanduser("~/.config/kaggle/kaggle.json")
os.makedirs(os.path.dirname(KAGGLE_DEST), exist_ok=True)

if os.path.exists(KAGGLE_JSON_PATH_ON_DRIVE):
    shutil.copy(KAGGLE_JSON_PATH_ON_DRIVE, KAGGLE_DEST)
    os.chmod(KAGGLE_DEST, 0o600)
    print("kaggle.json copied from Drive.")
else:
    print("WARNING: kaggle.json not found on Drive.")
    print("Place it at: /content/drive/MyDrive/kaggle.json")
    print("Or upload manually and set KAGGLE_DEST manually.")

# ── Dataset paths ──────────────────────────────────────────────────────────────
DATASET_ZIP   = "/content/openlogos.zip"
DATASET_DIR   = "/content/openlogos"
USE_CACHED    = False   # Set True to skip re-download if dataset already on Drive

CACHED_ZIP_ON_DRIVE = "/content/drive/MyDrive/detectra_models/openlogos.zip"

if USE_CACHED and os.path.exists(CACHED_ZIP_ON_DRIVE):
    print("Using cached dataset from Drive...")
    shutil.copy(CACHED_ZIP_ON_DRIVE, DATASET_ZIP)
else:
    print("Downloading OpenLogos-32 from Kaggle...")
    os.system(f"kaggle datasets download -d victorfrei/openlogos -p /content --unzip")
    # Also cache the zip to Drive for future runs
    if os.path.exists(DATASET_ZIP):
        shutil.copy(DATASET_ZIP, CACHED_ZIP_ON_DRIVE)
        print("Dataset cached to Drive.")

# Unzip if the directory doesn't exist yet
if not os.path.isdir(DATASET_DIR):
    if os.path.exists(DATASET_ZIP):
        print("Extracting dataset...")
        with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
            zf.extractall("/content")

# ── Discover dataset root ──────────────────────────────────────────────────────
# Kaggle may unzip into a sub-folder; find the folder containing class sub-dirs
def find_dataset_root(base: str) -> Path:
    """Walk up to 3 levels to find the first directory with 10+ sub-directories."""
    base_path = Path(base)
    for depth in range(4):
        candidates = list(base_path.rglob("*"))
        for p in candidates:
            if p.is_dir():
                subdirs = [x for x in p.iterdir() if x.is_dir()]
                if len(subdirs) >= 10:
                    return p
    return base_path

DATA_ROOT = find_dataset_root("/content")
print(f"Dataset root: {DATA_ROOT}")
classes_found = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"Classes found ({len(classes_found)}): {classes_found}")

## Step 4 — Dataset Setup with Augmentation

Augmentation strategy:
- **RandomResizedCrop(224)** — scale variation + spatial robustness
- **ColorJitter** — brightness, contrast, saturation, hue
- **RandomHorizontalFlip** — left-right invariance
- **RandomRotation(15°)** — orientation robustness
- ImageNet normalization (mean/std from ViT pretraining)

Validation uses only center-crop + normalization (no augmentation).

In [ ]:
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np

# ── Class list (must match LOGO_CLASSES in logo_recognizer.py) ─────────────────
LOGO_CLASSES = [
    "adidas", "apple", "bmw", "carlsberg", "chimay", "cocacola", "corona",
    "dhl", "erdinger", "esso", "fedex", "ferrari", "ford", "fosters",
    "google", "guiness", "heineken", "hp", "michelin", "minicooper",
    "nbc", "nike", "paulaner", "pepsi", "porsche", "puma", "redbull",
    "shell", "singha", "starbucks", "stellaartois", "texaco",
]
NUM_CLASSES = len(LOGO_CLASSES)
print(f"Number of classes: {NUM_CLASSES}")

# ── ImageNet stats (ViT was pretrained on ImageNet) ────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Load full dataset (train transform first; we'll override for val) ──────────
full_dataset = ImageFolder(root=str(DATA_ROOT))
print(f"Total images: {len(full_dataset)}")
print(f"Classes in dataset: {full_dataset.classes}")

# ── Verify class alignment ─────────────────────────────────────────────────────
dataset_classes_lower = [c.lower() for c in full_dataset.classes]
missing = set(LOGO_CLASSES) - set(dataset_classes_lower)
extra   = set(dataset_classes_lower) - set(LOGO_CLASSES)
if missing:
    print(f"WARNING — classes in LOGO_CLASSES but NOT in dataset: {missing}")
if extra:
    print(f"INFO — extra classes in dataset (will be ignored or included): {extra}")

# ── Train / Val split (80/20) ──────────────────────────────────────────────────
torch.manual_seed(42)
n_total = len(full_dataset)
n_val   = max(int(0.20 * n_total), NUM_CLASSES)   # at least 1 sample/class
n_train = n_total - n_val
train_indices, val_indices = random_split(
    range(n_total), [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

# Wrap with per-split transforms
class TransformSubset(torch.utils.data.Dataset):
    """Applies a transform to a subset of an ImageFolder dataset."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label

# ImageFolder returns PIL images before transform when we call it directly
# We need to bypass the dataset's own transform:
full_dataset.transform = None   # disable dataset-level transform
full_dataset.target_transform = None

train_dataset = TransformSubset(full_dataset, train_indices, train_transform)
val_dataset   = TransformSubset(full_dataset, val_indices,   val_transform)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## Step 5 — Model Setup

We load `google/vit-base-patch16-224` from HuggingFace Transformers and replace the
classifier head with a linear layer matching our 32 logo classes.

**Training schedule:**
- **Epochs 1–5:** Backbone frozen — only the classifier head is trained (fast convergence, avoids catastrophic forgetting)
- **Epochs 6–10:** Full model unfrozen — end-to-end fine-tuning at a lower effective LR via cosine decay

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTForImageClassification, ViTConfig

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Load pretrained ViT ────────────────────────────────────────────────────────
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,  # head size differs from pretrained 1000-class
    id2label={i: c for i, c in enumerate(LOGO_CLASSES)},
    label2id={c: i for i, c in enumerate(LOGO_CLASSES)},
)
model = model.to(DEVICE)

# ── Parameter count ────────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

# ── Freeze backbone (everything except classifier) ─────────────────────────────
def freeze_backbone(model):
    """Freeze all ViT encoder + embeddings; leave classifier trainable."""
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Backbone frozen. Trainable params: {trainable:,}")

def unfreeze_all(model):
    """Unfreeze the entire model for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Full model unfrozen. Trainable params: {trainable:,}")

freeze_backbone(model)
print("Model ready for phase-1 training (frozen backbone).")

## Step 5b — Optimizer, Scheduler & Training Loop

- **Optimizer:** AdamW, lr = 2e-4, weight decay = 0.01  
- **Scheduler:** Cosine annealing over all 10 epochs  
- **Loss:** CrossEntropyLoss  
- Checkpoints are saved to `/content/logo_checkpoints/` every epoch  
- Best model (by val accuracy) is tracked and saved separately

In [ ]:
import torch.optim as optim
import math
import time

LR           = 2e-4
WEIGHT_DECAY = 0.01
FREEZE_EPOCHS = 5
TOTAL_EPOCHS  = 10

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing over the total number of training steps
total_steps = TOTAL_EPOCHS * len(train_loader)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

# ── Gradient scaler for mixed-precision training (AMP) ─────────────────────────
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

# ── Training utilities ─────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, device=DEVICE):
    """Run one train or validation epoch. Returns (avg_loss, accuracy)."""
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for batch_idx, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                outputs = model(pixel_values=images)
                logits  = outputs.logits
                loss    = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            batch_size  = images.size(0)
            total_loss += loss.item() * batch_size
            preds       = logits.argmax(dim=-1)
            correct    += (preds == labels).sum().item()
            total      += batch_size

            if is_train and batch_idx % 20 == 0:
                print(f"  batch {batch_idx:4d}/{len(loader):4d} "
                      f"loss={loss.item():.4f} "
                      f"lr={scheduler.get_last_lr()[0]:.2e}",
                      end="\r", flush=True)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


# ── Main training loop ─────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc  = 0.0
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")

for epoch in range(1, TOTAL_EPOCHS + 1):
    t0 = time.time()

    # ── Phase transition at epoch FREEZE_EPOCHS + 1 ────────────────────────────
    if epoch == FREEZE_EPOCHS + 1:
        print("\n" + "="*60)
        print("Phase 2: Unfreezing backbone for full fine-tuning")
        print("="*60)
        unfreeze_all(model)
        # Re-create optimizer with all parameters
        optimizer = optim.AdamW(model.parameters(), lr=LR / 10, weight_decay=WEIGHT_DECAY)
        remaining_steps = (TOTAL_EPOCHS - FREEZE_EPOCHS) * len(train_loader)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=remaining_steps, eta_min=1e-7
        )

    print(f"\nEpoch {epoch}/{TOTAL_EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)

    elapsed = time.time() - t0
    print(f"\nEpoch {epoch} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
          f"time={elapsed:.0f}s")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # Save epoch checkpoint
    epoch_ckpt = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:02d}_acc{val_acc:.4f}.pt")
    torch.save(model.state_dict(), epoch_ckpt)

    # Track best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  ✓ New best model saved (val_acc={best_val_acc:.4f})")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

### Training Curves

Plot loss and accuracy over all epochs to check for overfitting or under-training.

In [ ]:
import matplotlib.pyplot as plt

epochs_x = range(1, TOTAL_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_x, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_x, history["val_loss"],   label="Val Loss",   marker="s")
axes[0].axvline(x=FREEZE_EPOCHS + 0.5, color="gray", linestyle="--", label="Unfreeze backbone")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_x, history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(epochs_x, history["val_acc"],   label="Val Acc",   marker="s")
axes[1].axvline(x=FREEZE_EPOCHS + 0.5, color="gray", linestyle="--", label="Unfreeze backbone")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("ViT Logo Recognition — Training History", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(GDRIVE_OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()
print("Training curves saved to Drive.")

## Step 6 — Evaluation: Accuracy & Confusion Matrix

Load the best checkpoint and run full evaluation on the validation set.
A normalized confusion matrix is plotted to highlight per-class performance and common misclassifications.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Load best checkpoint ───────────────────────────────────────────────────────
print(f"Loading best checkpoint: {best_ckpt_path}")
state_dict = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

# ── Collect predictions ────────────────────────────────────────────────────────
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
            outputs = model(pixel_values=images)
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Per-class accuracy ─────────────────────────────────────────────────────────
top1_accuracy = (all_preds == all_labels).mean()
print(f"\nTop-1 Validation Accuracy: {top1_accuracy:.4f} ({top1_accuracy*100:.2f}%)")

# Dataset class -> LOGO_CLASSES index mapping
dataset_class_to_idx = full_dataset.class_to_idx
idx_to_logo = {}   # dataset label index -> LOGO_CLASSES display name
for cls_name, ds_idx in dataset_class_to_idx.items():
    logo_name = cls_name.lower()
    if logo_name in LOGO_CLASSES:
        idx_to_logo[ds_idx] = logo_name

# Per-class accuracy
print("\nPer-class accuracy:")
for cls_idx in sorted(idx_to_logo.keys()):
    cls_name  = idx_to_logo[cls_idx]
    mask      = all_labels == cls_idx
    if mask.sum() == 0:
        continue
    cls_acc   = (all_preds[mask] == cls_idx).mean()
    count     = mask.sum()
    print(f"  {cls_name:15s} {cls_acc*100:6.1f}%  ({count} samples)")

# ── Confusion Matrix ───────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix   # pre-installed on Colab

# Only include class indices that appear in the dataset
present_indices = sorted(set(all_labels.tolist()))
present_names   = [idx_to_logo.get(i, f"cls_{i}") for i in present_indices]

cm = confusion_matrix(all_labels, all_preds, labels=present_indices)
# Normalize row-wise
cm_norm = cm.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1   # avoid division by zero
cm_norm /= row_sums

n = len(present_names)
fig_size = max(12, n // 2)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(present_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(present_names, fontsize=9)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title(f"Confusion Matrix (Normalized) — Val Acc {top1_accuracy*100:.1f}%", fontsize=13)

thresh = 0.5
for i in range(n):
    for j in range(n):
        val = cm_norm[i, j]
        color = "white" if val > thresh else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color=color)

plt.tight_layout()
cm_path = os.path.join(GDRIVE_OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved to: {cm_path}")

## Step 7 — Export to ONNX

Export the best PyTorch model to ONNX format with a **dynamic batch dimension**.
The exported model accepts `pixel_values` of shape `(N, 3, 224, 224)` and outputs
logits of shape `(N, 32)`.

- Opset 17 is used for maximum compatibility with onnxruntime 1.18
- Dynamic axes allow any batch size at inference time

In [ ]:
import torch
import onnx

ONNX_FP32_PATH = "/content/logo_vit_fp32.onnx"

# ── Ensure best model is loaded ────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
model.eval()
model.to("cpu")   # Export from CPU for broadest compatibility

# ── Create a dummy input (batch=1, RGB, 224×224) ───────────────────────────────
dummy_input = torch.randn(1, 3, 224, 224, requires_grad=False)

# ── Wrapper: ViTForImageClassification takes keyword args ──────────────────────
class ViTONNXWrapper(torch.nn.Module):
    """Thin wrapper so torch.onnx.export sees a positional-arg model."""
    def __init__(self, vit_model):
        super().__init__()
        self.model = vit_model

    def forward(self, pixel_values):
        return self.model(pixel_values=pixel_values).logits

wrapper = ViTONNXWrapper(model)
wrapper.eval()

# ── Export ─────────────────────────────────────────────────────────────────────
print("Exporting model to ONNX...")
torch.onnx.export(
    wrapper,
    dummy_input,
    ONNX_FP32_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["pixel_values"],
    output_names=["logits"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "logits":       {0: "batch_size"},
    },
)

# ── Verify exported model ──────────────────────────────────────────────────────
onnx_model = onnx.load(ONNX_FP32_PATH)
onnx.checker.check_model(onnx_model)
onnx_size_mb = os.path.getsize(ONNX_FP32_PATH) / 1e6
print(f"ONNX export complete. File size: {onnx_size_mb:.1f} MB")

# ── Quick sanity check with onnxruntime ───────────────────────────────────────
import onnxruntime as ort

sess_fp32 = ort.InferenceSession(ONNX_FP32_PATH, providers=["CPUExecutionProvider"])
test_input = dummy_input.numpy()
logits_ort = sess_fp32.run(None, {"pixel_values": test_input})[0]
logits_pt  = wrapper(dummy_input).detach().numpy()

max_diff = abs(logits_ort - logits_pt).max()
print(f"Max logit difference (PyTorch vs ONNX): {max_diff:.6f}")
assert max_diff < 1e-3, f"ONNX export mismatch too large: {max_diff}"
print("Numerical sanity check PASSED.")

# Move model back to GPU for any further evaluation
model.to(DEVICE)

## Step 8 — INT8 Dynamic Quantization

Apply post-training dynamic quantization using `onnxruntime.quantization.quantize_dynamic`.
This quantizes weight matrices to INT8, reducing model size by ~4× and typically
improving CPU inference speed by 2–3×, with minimal accuracy loss.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

ONNX_INT8_PATH = "/content/logo_vit_quantized.onnx"

print("Applying INT8 dynamic quantization...")
quantize_dynamic(
    model_input=ONNX_FP32_PATH,
    model_output=ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,
    per_channel=False,   # per-channel can improve accuracy but may reduce speed
    reduce_range=False,  # set True only on older AVX2 hardware
)

fp32_mb = os.path.getsize(ONNX_FP32_PATH) / 1e6
int8_mb = os.path.getsize(ONNX_INT8_PATH) / 1e6
print(f"FP32 model: {fp32_mb:.1f} MB")
print(f"INT8 model: {int8_mb:.1f} MB")
print(f"Compression ratio: {fp32_mb / int8_mb:.2f}×")

# ── Validate INT8 model accuracy on val set ────────────────────────────────────
sess_int8 = ort.InferenceSession(ONNX_INT8_PATH, providers=["CPUExecutionProvider"])

print("Evaluating INT8 model on validation set...")
int8_correct = 0
int8_total   = 0

for images, labels in val_loader:
    # ViT needs float32 input even for INT8 quantized weight model
    inp = images.numpy().astype(np.float32)
    logits = sess_int8.run(None, {"pixel_values": inp})[0]
    preds  = logits.argmax(axis=-1)
    int8_correct += (preds == labels.numpy()).sum()
    int8_total   += len(labels)

int8_acc = int8_correct / int8_total
print(f"INT8 Validation Accuracy: {int8_acc:.4f} ({int8_acc*100:.2f}%)")
print(f"Accuracy drop vs FP32:    {(top1_accuracy - int8_acc)*100:.2f} pp")

## Step 9 — CPU Inference Speed Benchmark

Benchmark the INT8 quantized model on CPU to verify it meets the latency budget
for real-time video analysis (~30 FPS = 33 ms/frame).

We measure:
- **Latency** (ms/image) for batch size 1
- **Throughput** (images/sec) for batch size 32

In [ ]:
import time
import onnxruntime as ort
import numpy as np

# Configure session options for CPU benchmark
sess_options = ort.SessionOptions()
sess_options.inter_op_num_threads = 4
sess_options.intra_op_num_threads = 4
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

bench_sess = ort.InferenceSession(
    ONNX_INT8_PATH,
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)

def benchmark_ort(session, batch_size=1, n_warmup=20, n_iters=100):
    """Returns mean latency in ms."""
    dummy = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)
    feed  = {"pixel_values": dummy}

    # Warmup
    for _ in range(n_warmup):
        session.run(None, feed)

    # Timed runs
    times = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        session.run(None, feed)
        times.append((time.perf_counter() - t0) * 1000)

    return {
        "mean_ms":   np.mean(times),
        "median_ms": np.median(times),
        "p95_ms":    np.percentile(times, 95),
        "min_ms":    np.min(times),
    }

print("Benchmarking INT8 model on CPU...")
print("="*50)

# Single image latency
stats_1 = benchmark_ort(bench_sess, batch_size=1)
print(f"Batch size = 1")
print(f"  Mean latency:   {stats_1['mean_ms']:.1f} ms")
print(f"  Median latency: {stats_1['median_ms']:.1f} ms")
print(f"  P95 latency:    {stats_1['p95_ms']:.1f} ms")
print(f"  Min latency:    {stats_1['min_ms']:.1f} ms")
fps_1 = 1000 / stats_1['mean_ms']
print(f"  Throughput:     {fps_1:.1f} FPS")

print()

# Batch=32 throughput
stats_32 = benchmark_ort(bench_sess, batch_size=32)
fps_32 = 32 * 1000 / stats_32['mean_ms']
print(f"Batch size = 32")
print(f"  Mean latency:   {stats_32['mean_ms']:.1f} ms / batch")
print(f"  Throughput:     {fps_32:.1f} images/sec")

print()
print("="*50)
target_ms = 33.0
status = "PASS" if stats_1['mean_ms'] < target_ms else "FAIL"
print(f"Real-time target (<{target_ms}ms): {status} ({stats_1['mean_ms']:.1f}ms)")

## Step 10 — Save Model to Google Drive

Copy the quantized ONNX model and class label mapping to Google Drive for
persistent storage. The model can then be copied to your Detectra backend.

In [ ]:
import shutil, json

# ── Final output paths ─────────────────────────────────────────────────────────
GDRIVE_QUANTIZED_PATH  = os.path.join(GDRIVE_OUTPUT_DIR, "logo_vit_quantized.onnx")
GDRIVE_FP32_PATH       = os.path.join(GDRIVE_OUTPUT_DIR, "logo_vit_fp32.onnx")
GDRIVE_LABELS_PATH     = os.path.join(GDRIVE_OUTPUT_DIR, "logo_classes.json")
GDRIVE_METRICS_PATH    = os.path.join(GDRIVE_OUTPUT_DIR, "training_metrics.json")

# ── Save quantized model ───────────────────────────────────────────────────────
shutil.copy(ONNX_INT8_PATH, GDRIVE_QUANTIZED_PATH)
print(f"Quantized model saved: {GDRIVE_QUANTIZED_PATH}")
print(f"  Size: {os.path.getsize(GDRIVE_QUANTIZED_PATH)/1e6:.1f} MB")

# ── Save FP32 model (for potential future use) ────────────────────────────────
shutil.copy(ONNX_FP32_PATH, GDRIVE_FP32_PATH)
print(f"FP32 model saved:      {GDRIVE_FP32_PATH}")

# ── Save class labels JSON ─────────────────────────────────────────────────────
labels_data = {
    "logo_classes": LOGO_CLASSES,
    "num_classes":  NUM_CLASSES,
    "id2label":     {str(i): c for i, c in enumerate(LOGO_CLASSES)},
    "label2id":     {c: i for i, c in enumerate(LOGO_CLASSES)},
}
with open(GDRIVE_LABELS_PATH, "w") as f:
    json.dump(labels_data, f, indent=2)
print(f"Class labels saved:    {GDRIVE_LABELS_PATH}")

# ── Save training metrics ──────────────────────────────────────────────────────
metrics = {
    "best_val_accuracy_fp32": float(top1_accuracy),
    "best_val_accuracy_int8": float(int8_acc),
    "accuracy_drop_pp":       float((top1_accuracy - int8_acc) * 100),
    "fp32_model_mb":          round(fp32_mb, 2),
    "int8_model_mb":          round(int8_mb, 2),
    "compression_ratio":      round(fp32_mb / int8_mb, 2),
    "cpu_latency_ms_bs1":     round(stats_1['mean_ms'], 2),
    "cpu_fps_bs1":            round(fps_1, 1),
    "cpu_throughput_bs32":    round(fps_32, 1),
    "total_epochs":           TOTAL_EPOCHS,
    "freeze_epochs":          FREEZE_EPOCHS,
    "batch_size":             BATCH_SIZE,
    "learning_rate":          LR,
    "history":                history,
}
with open(GDRIVE_METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Training metrics saved: {GDRIVE_METRICS_PATH}")

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TRAINING COMPLETE — SUMMARY")
print("="*60)
print(f"  FP32 Val Accuracy:  {top1_accuracy*100:.2f}%")
print(f"  INT8 Val Accuracy:  {int8_acc*100:.2f}%")
print(f"  Accuracy drop:      {(top1_accuracy-int8_acc)*100:.2f} percentage points")
print(f"  INT8 model size:    {int8_mb:.1f} MB")
print(f"  CPU latency (bs=1): {stats_1['mean_ms']:.1f} ms")
print()
print("FILES SAVED TO GOOGLE DRIVE:")
print(f"  {GDRIVE_QUANTIZED_PATH}")
print(f"  {GDRIVE_FP32_PATH}")
print(f"  {GDRIVE_LABELS_PATH}")
print(f"  {GDRIVE_METRICS_PATH}")
print(f"  {GDRIVE_OUTPUT_DIR}/training_curves.png")
print(f"  {GDRIVE_OUTPUT_DIR}/confusion_matrix.png")
print()
print("NEXT STEP:")
print("  Copy logo_vit_quantized.onnx to:")
print("  detectra-ai/backend/models/logo_vit_quantized.onnx")

## Appendix — Deployment Notes

### Using the model in Detectra

1. Copy `logo_vit_quantized.onnx` from Drive to `detectra-ai/backend/models/`
2. The `LogoRecognizerService` in `logo_recognizer.py` will automatically pick it up
3. Inference uses `CPUExecutionProvider` with 4 threads (configurable in `logo_recognizer.py`)

### Preprocessing (must match training)

```python
# Resize to 224×224
# Convert BGR→RGB, normalize to [0,1]
# Apply ImageNet mean/std:
#   mean = [0.485, 0.456, 0.406]
#   std  = [0.229, 0.224, 0.225]
# Transpose to NCHW: (1, 3, 224, 224)
```

### Class order

The model's output logit index `i` maps to `LOGO_CLASSES[i]`:

```python
LOGO_CLASSES = [
    "adidas", "apple", "bmw", "carlsberg", "chimay", "cocacola", "corona",
    "dhl", "erdinger", "esso", "fedex", "ferrari", "ford", "fosters",
    "google", "guiness", "heineken", "hp", "michelin", "minicooper",
    "nbc", "nike", "paulaner", "pepsi", "porsche", "puma", "redbull",
    "shell", "singha", "starbucks", "stellaartois", "texaco",
]
```

### Re-training tips

- Increase `TOTAL_EPOCHS` to 15–20 if val accuracy < 85%
- Add `RandomGrayscale(p=0.1)` to augmentations for grayscale logo robustness
- Use `per_channel=True` in `quantize_dynamic` for slightly better INT8 accuracy
- If GPU OOM: reduce `BATCH_SIZE` to 16 and enable gradient checkpointing